# Assignment 2: training pipeline (hard hat compliance)

This notebook runs start to finish with `uv run python main.py train`. These are the steps it goes through:
1. Download the Hard Hat Workers dataset from Roboflow Universe.
2. Describe the dataset (instances per class, split sizes and object size).
3. Finetune YOLO11n from the pretrained checkpoint and save the best weights to `weights/`.
4. Evaluate it on the test split against the COCO model used zero-shot and a model that predicts nothing.
5. Score every image, sweep the threshold, put a cost on it, and choose the threshold and the review band.
6. Save everything the dashboard needs to `artifacts/`, including a few test images.

For a quick test run you can set `EPOCHS=1 FRACTION=0.05`. If I only change the costs, `SKIP_TRAIN=1` reuses `weights/best.pt` so I don't have to train again.

In [ ]:
import json
import os
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import yaml
from roboflow import Roboflow
from ultralytics import YOLO
from ultralytics.utils.metrics import ap_per_class, box_iou

import costing
from api_key import load_api_key

ROOT = Path.cwd()
DATA_DIR = ROOT / "data"
RUNS_DIR = ROOT / "runs"
WEIGHTS_DIR = ROOT / "weights"
ARTIFACTS = ROOT / "artifacts"
for d in (DATA_DIR, WEIGHTS_DIR, ARTIFACTS):
    d.mkdir(exist_ok=True)

RF_WORKSPACE, RF_PROJECT, RF_VERSION = "joseph-nelson", "hard-hat-workers", 10

BASE_MODEL = "yolo11n.pt"
COCO_MODEL = "yolo11n.pt"  # same checkpoint, used zero-shot as the baseline
EPOCHS = int(os.environ.get("EPOCHS", 20))
FRACTION = float(os.environ.get("FRACTION", 1.0))  # share of the train split; < 1 only for smoke runs
SKIP_TRAIN = os.environ.get("SKIP_TRAIN") == "1"  # reuse weights/best.pt, e.g. after changing costs
RESUME = os.environ.get("RESUME") == "1"  # continue an interrupted run from runs/hardhat/weights/last.pt
IMGSZ, BATCH, PATIENCE, SEED = 640, 16, 8, 0
DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"

POSITIVE = "head"  # a bare head: a worker without a helmet
print(f"Device: {DEVICE}, epochs: {EPOCHS}, fraction: {FRACTION}")

## Costs and operating assumptions

One case is one camera snapshot of a work area, and it is positive if at least one worker in it has a bare head. A missed bare head costs the expected fine (one OSHA serious violation times the chance it gets caught and fined) plus the expected injury cost (an average injury claim times the chance of a head injury). I explain where every number comes from in REPORT.md.

In [ ]:
FINE_USD = 16550.0    # OSHA serious violation, per worker
P_FINE = 0.10         # chance a missed bare head ends up caught and fined
INJURY_USD = 42000.0  # NSC average cost of a workplace injury needing medical attention
P_INJURY = 0.01       # chance a missed bare head ends in a head injury
USD_TO_EUR = 0.86     # exchange rate assumed

COSTS = {  # euros per case
    "fn": round((FINE_USD * P_FINE + INJURY_USD * P_INJURY) * USD_TO_EUR, 2),  # bare head missed: expected fine + injury
    "fp": 10.0,     # false alarm, ~15 min of a supervisor's time walking over for nothing
    "tp": 0.0,      # correct alarm, the supervisor is just doing their job
    "tn": 0.0,      # correctly left alone
    "review": 0.5,  # a person looking at one flagged snapshot for ~30 s
}
BASE_RATE = 0.05  # share of snapshots that really contain a bare head

OPS = {
    "cases_per_day": 2000,           # 10 cameras, one snapshot every 3 min, 10 h shift
    "review_minutes": 0.5,           # minutes to check one snapshot
    "reviewer_minutes_per_day": 420, # productive minutes in one reviewer's day
    "reviewers_available": 1,        # people you can actually staff
}
print(f"FN cost: EUR {COSTS['fn']:,.2f}")

## 1. Download the dataset

In [ ]:
def download_dataset():
    location = DATA_DIR / f"{RF_PROJECT}-{RF_VERSION}"
    if (location / "data.yaml").exists() and (location / "test" / "images").is_dir():
        print("Dataset already downloaded, reusing", location)
        return location
    rf = Roboflow(api_key=load_api_key())
    version = rf.workspace(RF_WORKSPACE).project(RF_PROJECT).version(RF_VERSION)
    dataset = version.download("yolov11", location=str(location))
    return Path(dataset.location)


def write_data_yaml(root):
    """Roboflow's data.yaml uses relative paths that break outside its folder; rewrite them absolute."""
    names = yaml.safe_load((root / "data.yaml").read_text())["names"]
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names)]
    cfg = {"path": str(root), "train": "train/images", "val": "valid/images", "test": "test/images", "names": names}
    path = root / "data_abs.yaml"
    path.write_text(yaml.safe_dump(cfg))
    return path, names


dataset_root = download_dataset()
data_yaml, NAMES = write_data_yaml(dataset_root)
POS_ID = NAMES.index(POSITIVE)
print(dataset_root, NAMES)

## 2. Describe the dataset

In [ ]:
def load_split(split):
    """One item per image: path, ground-truth classes and normalised xyxy boxes."""
    split_dir = dataset_root / split
    items = []
    for img in sorted((split_dir / "images").iterdir()):
        label = split_dir / "labels" / f"{img.stem}.txt"
        rows = [line.split() for line in label.read_text().splitlines() if line.strip()] if label.exists() else []
        rows = np.array([[float(v) for v in r[:5]] for r in rows], dtype=np.float32).reshape(-1, 5)
        cx, cy, w, h = rows[:, 1], rows[:, 2], rows[:, 3], rows[:, 4]
        items.append({
            "image": img,
            "cls": rows[:, 0].astype(int),
            "boxes": np.stack([cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2], axis=1),
            "area": w * h,
        })
    return items


splits = {s: load_split(s) for s in ("train", "valid", "test")}


def describe(items):
    out = {"images": len(items), "images_without_labels": sum(len(it["cls"]) == 0 for it in items), "classes": {}}
    for c, name in enumerate(NAMES):
        areas = np.concatenate([it["area"][it["cls"] == c] for it in items])
        out["classes"][name] = {
            "instances": int(len(areas)),
            "images_containing": int(sum((it["cls"] == c).any() for it in items)),
            "median_rel_area": float(np.median(areas)) if len(areas) else None,
            "p10_rel_area": float(np.percentile(areas, 10)) if len(areas) else None,
            "p90_rel_area": float(np.percentile(areas, 90)) if len(areas) else None,
            "share_under_1pct_of_image": float(np.mean(areas < 0.01)) if len(areas) else None,
        }
    return out


dataset_stats = {s: describe(items) for s, items in splits.items()}
(ARTIFACTS / "dataset_stats.json").write_text(json.dumps(dataset_stats, indent=2))
for s, st in dataset_stats.items():
    print(f"\n{s}: {st['images']} images ({st['images_without_labels']} unlabelled)")
    print(pd.DataFrame(st["classes"]).T.to_string())

## 3. Finetune

In [ ]:
best_path = WEIGHTS_DIR / "best.pt"
if SKIP_TRAIN and best_path.exists():
    print("SKIP_TRAIN=1: reusing", best_path)
elif RESUME and (RUNS_DIR / "hardhat" / "weights" / "last.pt").exists():
    model = YOLO(str(RUNS_DIR / "hardhat" / "weights" / "last.pt"))
    model.train(resume=True)
    shutil.copy(model.trainer.best, best_path)
    print("Resumed and saved", best_path)
else:
    model = YOLO(BASE_MODEL)
    model.train(
        data=str(data_yaml), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, patience=PATIENCE,
        fraction=FRACTION, device=DEVICE, seed=SEED, project=str(RUNS_DIR), name="hardhat", exist_ok=True,
    )
    shutil.copy(model.trainer.best, best_path)
    print("Saved", best_path)
finetuned = YOLO(str(best_path))

## 4. Evaluate against the baselines

All three models are scored with the same code on the same test split. The COCO model is the pretrained YOLO11n with no finetuning. It was trained on 80 everyday classes, which include `person` but not `head` or `helmet`, so the only thing I can compare is its `person` predictions and it scores zero on the two classes my decision needs. That is expected, and it shows how much the finetuning adds. The trivial baseline predicts nothing.

In [ ]:
IOUV = np.linspace(0.5, 0.95, 10)


def predict(model, items, class_map=None):
    """Low-confidence predictions for every image, so later thresholds can be swept."""
    preds = []
    paths = [str(it["image"]) for it in items]
    # A list source is one batch to Ultralytics, so feed it in chunks or the GPU runs out of memory
    results = (r for i in range(0, len(paths), 16)
               for r in model.predict(paths[i:i + 16], conf=0.001, imgsz=IMGSZ, device=DEVICE, verbose=False))
    for r in results:
        cls = r.boxes.cls.cpu().numpy().astype(int)
        boxes = r.boxes.xyxyn.cpu().numpy()
        conf = r.boxes.conf.cpu().numpy()
        if class_map is not None:
            keep = np.isin(cls, list(class_map))
            cls = np.array([class_map[c] for c in cls[keep]], dtype=int)
            boxes, conf = boxes[keep], conf[keep]
        preds.append({"cls": cls, "boxes": boxes.reshape(-1, 4), "conf": conf})
    return preds


def iou_matrix(gt_boxes, pred_boxes):
    return box_iou(torch.as_tensor(gt_boxes, dtype=torch.float32),
                   torch.as_tensor(pred_boxes, dtype=torch.float32)).numpy()


def detection_metrics(items, preds):
    """Per-class precision, recall, AP50, AP50-95, matched the way Ultralytics does it."""
    tp, conf, pred_cls, target_cls = [], [], [], []
    for it, p in zip(items, preds):
        correct = np.zeros((len(p["cls"]), len(IOUV)), dtype=bool)
        if len(p["cls"]) and len(it["cls"]):
            iou = iou_matrix(it["boxes"], p["boxes"]) * (it["cls"][:, None] == p["cls"][None, :])
            for k, t in enumerate(IOUV):
                m = np.argwhere(iou >= t)
                if len(m):
                    m = m[iou[m[:, 0], m[:, 1]].argsort()[::-1]]
                    m = m[np.unique(m[:, 1], return_index=True)[1]]
                    m = m[np.unique(m[:, 0], return_index=True)[1]]
                    correct[m[:, 1], k] = True
        tp.append(correct); conf.append(p["conf"]); pred_cls.append(p["cls"]); target_cls.append(it["cls"])
    tp, conf, pred_cls, target_cls = (np.concatenate(x) for x in (tp, conf, pred_cls, target_cls))

    per_class = {n: {"instances": int((target_cls == c).sum()), "precision": 0.0, "recall": 0.0,
                     "ap50": 0.0, "ap50_95": 0.0} for c, n in enumerate(NAMES)}
    if len(conf):
        _, _, p, r, _, ap, classes, *_ = ap_per_class(tp, conf, pred_cls, target_cls)
        for i, c in enumerate(classes):
            per_class[NAMES[c]].update(precision=float(p[i]), recall=float(r[i]),
                                       ap50=float(ap[i, 0]), ap50_95=float(ap[i].mean()))
    present = [v for v in per_class.values() if v["instances"]]
    overall = {"map50": float(np.mean([v["ap50"] for v in present])),
               "map50_95": float(np.mean([v["ap50_95"] for v in present]))}
    return {"overall": overall, "per_class": per_class}


def confusion(items, preds, conf_thr=0.25, iou_thr=0.45):
    """Rows = true class (+ background), columns = predicted class (+ missed)."""
    nc = len(NAMES)
    cm = np.zeros((nc + 1, nc + 1), dtype=int)
    for it, p in zip(items, preds):
        keep = p["conf"] >= conf_thr
        pb, pc = p["boxes"][keep], p["cls"][keep]
        used_g, used_p = set(), set()
        if len(pb) and len(it["cls"]):
            iou = iou_matrix(it["boxes"], pb)
            pairs = np.argwhere(iou >= iou_thr)
            for g, q in pairs[iou[pairs[:, 0], pairs[:, 1]].argsort()[::-1]]:
                if g not in used_g and q not in used_p:
                    used_g.add(g); used_p.add(q)
                    cm[it["cls"][g], pc[q]] += 1
        for g in set(range(len(it["cls"]))) - used_g:
            cm[it["cls"][g], nc] += 1
        for q in set(range(len(pb))) - used_p:
            cm[nc, pc[q]] += 1
    return cm


coco = YOLO(COCO_MODEL)
coco_map = {0: NAMES.index("person")} if "person" in NAMES else {}

preds = {
    "test": {"finetuned": predict(finetuned, splits["test"]),
             "coco_zero_shot": predict(coco, splits["test"], coco_map)},
    "valid": {"finetuned": predict(finetuned, splits["valid"]),
              "coco_zero_shot": predict(coco, splits["valid"], coco_map)},
}
nothing = [{"cls": np.zeros(0, int), "boxes": np.zeros((0, 4)), "conf": np.zeros(0)} for _ in splits["test"]]

detection = {
    "finetuned": detection_metrics(splits["test"], preds["test"]["finetuned"]),
    "coco_zero_shot": detection_metrics(splits["test"], preds["test"]["coco_zero_shot"]),
    "predict_nothing": detection_metrics(splits["test"], nothing),
}
cm = confusion(splits["test"], preds["test"]["finetuned"])
(ARTIFACTS / "detection_metrics.json").write_text(json.dumps(detection, indent=2))
(ARTIFACTS / "confusion_matrix.json").write_text(json.dumps(
    {"labels": NAMES + ["background"], "matrix": cm.tolist(), "conf": 0.25, "iou": 0.45,
     "note": "rows = true, columns = predicted; background column = missed, background row = false detection"},
    indent=2))

for name, d in detection.items():
    print(f"\n{name}: mAP50 {d['overall']['map50']:.3f}, mAP50-95 {d['overall']['map50_95']:.3f}")
    print(pd.DataFrame(d["per_class"]).T.round(3).to_string())
print("\nConfusion matrix (rows true, cols predicted):")
print(pd.DataFrame(cm, index=NAMES + ["background"], columns=NAMES + ["missed"]).to_string())

## 5. Price the decision

Each image gets one score, which is the highest `head` confidence in it. I choose the threshold and the review band on the validation split and then report them on the test split, so they are not tuned on the same numbers they are judged by. The COCO model cannot see heads, so for its best attempt I score each image by its highest `person` confidence.

In [ ]:
def image_scores(items, preds, cls_id):
    rows = []
    for it, p in zip(items, preds):
        hits = p["conf"][p["cls"] == cls_id]
        rows.append({"image": it["image"].name, "y_true": int((it["cls"] == POS_ID).any()),
                     "score": float(hits.max()) if len(hits) else 0.0})
    return pd.DataFrame(rows)


scores = {}
for split in ("valid", "test"):
    ft = image_scores(splits[split], preds[split]["finetuned"], POS_ID)
    cz = image_scores(splits[split], preds[split]["coco_zero_shot"], coco_map.get(0, -1))
    ft["score_coco"] = cz["score"]
    ft.to_csv(ARTIFACTS / f"scores_{split}.csv", index=False)
    scores[split] = ft

val, test = scores["valid"], scores["test"]
sweep_val = costing.sweep(val["score"], val["y_true"], COSTS, BASE_RATE)
sweep_test = costing.sweep(test["score"], test["y_true"], COSTS, BASE_RATE)
sweep_coco = costing.sweep(test["score_coco"], test["y_true"], COSTS, BASE_RATE)
sweep = sweep_val.merge(sweep_test, on="threshold", suffixes=("_val", "_test"))
sweep["cost_per_1000_coco_test"] = sweep_coco["cost_per_1000"]
sweep.to_csv(ARTIFACTS / "cost_sweep.csv", index=False)

threshold = float(sweep_val.loc[sweep_val["cost_per_1000"].idxmin(), "threshold"])
band = costing.best_band(val["score"], val["y_true"], COSTS, BASE_RATE, OPS)

model_alone_test, _ = costing.band_cost(test["score"], test["y_true"], threshold, threshold, COSTS, BASE_RATE)
band_test, review_share_test = costing.band_cost(test["score"], test["y_true"], band["low"], band["high"], COSTS, BASE_RATE)
alts = costing.alternatives(COSTS, BASE_RATE)

operating_point = {
    "positive_class": POSITIVE,
    "threshold": threshold,
    "band": {"low": band["low"], "high": band["high"]},
    "test_cost_per_1000": {
        "do_nothing": alts["do_nothing"],
        "person_checks_everything": alts["person_checks_everything"],
        "model_alone": model_alone_test,
        "model_plus_review": band_test,
        "coco_zero_shot_best": float(sweep_coco["cost_per_1000"].min()),
    },
    "review_share_test": review_share_test,
    "reviews_per_day": review_share_test * OPS["cases_per_day"],
    "reviewers_needed": costing.reviewers_needed(review_share_test, OPS),
    "reviewers_needed_check_everything": costing.reviewers_needed(1.0, OPS),
}
config = {
    "dataset": {"workspace": RF_WORKSPACE, "project": RF_PROJECT, "version": RF_VERSION, "classes": NAMES},
    "training": {"base_model": BASE_MODEL, "epochs": EPOCHS, "fraction": FRACTION, "imgsz": IMGSZ,
                 "batch": BATCH, "patience": PATIENCE, "seed": SEED, "device": DEVICE},
    "costs": COSTS, "base_rate": BASE_RATE, "ops": OPS,
}
(ARTIFACTS / "operating_point.json").write_text(json.dumps(operating_point, indent=2))
(ARTIFACTS / "config.json").write_text(json.dumps(config, indent=2))
print(json.dumps(operating_point, indent=2))

## 6. Sample test images for the dashboard

`data/` is never committed, so on anyone else's machine the dashboard would have no images to show. This copies two test images for every truth and outcome combination of the chosen band (pass, review and fail, right and wrong) into `artifacts/examples/`, so the mistakes can be looked at and not just counted.

In [ ]:
EXAMPLES_DIR = ARTIFACTS / "examples"
shutil.rmtree(EXAMPLES_DIR, ignore_errors=True)
EXAMPLES_DIR.mkdir()

outcome = np.where(test["score"] >= band["high"], "fail", np.where(test["score"] < band["low"], "pass", "review"))
examples = (test.assign(outcome=outcome).sample(frac=1, random_state=SEED)
            .groupby(["y_true", "outcome"]).head(2).sort_values(["y_true", "outcome"]))
for name in examples["image"]:
    shutil.copy(dataset_root / "test" / "images" / name, EXAMPLES_DIR / name)
examples.to_csv(EXAMPLES_DIR / "examples.csv", index=False)
print(examples[["image", "y_true", "outcome", "score"]].to_string(index=False))